# Getting Started

Welcome to jax-healpy! This guide will help you get up and running with JAX-based HEALPix functions.

## Basic Usage

### Import and Setup

In [ ]:
import jax
import jax.numpy as jnp

import jax_healpy as hp

# Enable 64-bit precision (recommended, see the note below)
jax.config.update('jax_enable_x64', True)

!!! note
    jax-healpy follows JAX's precision setting and does not enable 64-bit precision on import. With the
    default 32-bit precision, the results diverge from healpy even at moderate resolution (e.g. `nside = 256`).
    64-bit precision should be enabled before the first array operation.

### HEALPix Basics

In [ ]:
# Set up a HEALPix map
nside = 64
npix = hp.nside2npix(nside)
print(f'Number of pixels: {npix}')

# Create some test data
pixels = jnp.arange(npix)

### Coordinate Conversions

In [ ]:
# Convert pixel indices to angular coordinates
theta, phi = hp.pix2ang(nside, pixels)

# Convert back to pixels
recovered_pixels = hp.ang2pix(nside, theta, phi)

# Verify the conversion
print(f'Conversion successful: {jnp.array_equal(pixels, recovered_pixels)}')

### Vector Operations

In [ ]:
# Convert pixels to 3D unit vectors, of shape (npix, 3)
vectors = hp.pix2vec(nside, pixels)

# Convert vectors back to pixels
recovered_pixels_vec = hp.vec2pix(nside, vectors[:, 0], vectors[:, 1], vectors[:, 2])

print(f'Vector conversion successful: {jnp.array_equal(pixels, recovered_pixels_vec)}')

### Pixel Ordering Schemes

In [ ]:
# Convert between RING and NESTED schemes
ring_pixels = jnp.arange(100)  # First 100 pixels in RING scheme
nest_pixels = hp.ring2nest(nside, ring_pixels)
back_to_ring = hp.nest2ring(nside, nest_pixels)

print(f'Scheme conversion successful: {jnp.array_equal(ring_pixels, back_to_ring)}')

## Working with Maps

### Map Interpolation

In [ ]:
# Create a simple test map
test_map = jnp.sin(theta) * jnp.cos(phi)

# Get the interpolation pixels and weights for arbitrary coordinates
theta_interp = jnp.array([0.5, 1.0, 1.5])
phi_interp = jnp.array([0.0, 1.0, 2.0])
neighbours, weights = hp.get_interp_weights(nside, theta_interp, phi_interp)

# Interpolate values
interpolated_values = hp.get_interp_val(test_map, theta_interp, phi_interp)

jax-healpy does not provide visualization functions: we display the maps with healpy's
`mollview`, in the Mollweide projection. The interpolation points are marked with crosses.

In [ ]:
import matplotlib.pyplot as plt
from healpy import mollview, projscatter

mollview(test_map, title='sin(theta) cos(phi)', cmap='RdBu_r', min=-1, max=1)
projscatter(theta_interp, phi_interp, marker='x', s=80, color='black')
plt.show()

print(f'Interpolated values: {interpolated_values}')

### Spherical Harmonics

The spherical harmonic transforms require the `cpu` or the `gpu` extra dependencies, see the [installation](installation.md#spherical-harmonic-transforms).

In [ ]:
lmax = 2 * nside

# Forward transform: map to spherical harmonic coefficients
alm = hp.map2alm(test_map, lmax=lmax)

# Inverse transform: coefficients back to map
reconstructed_map = hp.alm2map(alm, nside=nside, lmax=lmax)

print(f'SHT round-trip error: {jnp.max(jnp.abs(test_map - reconstructed_map))}')

This smooth map is reconstructed almost exactly: the error, of the order of $10^{-9}$, is
concentrated in the polar caps, where the HEALPix pixels are the least regular.

In [ ]:
residual = reconstructed_map - test_map
vmax = float(jnp.abs(residual).max())
mollview(residual, title='alm2map(map2alm(map)) - map', cmap='RdBu_r', min=-vmax, max=vmax)
plt.show()